Importing necessary libraries and the data

In [1]:
from IPython.display import display

from config import DATA_DIR
import pandas as pd
from datetime import datetime

df = pd.read_excel(DATA_DIR / "dataset_exercise.xlsx")
df.head()



FileNotFoundError: [Errno 2] No such file or directory: '../data/dataset_exercise.xlsx'

In [ ]:
df.info()

Check to make sure the alarms are in order

In [ ]:
if df["start_alarm"].index.is_monotonic_increasing != True:
    # not necessary as the start_alarm values are already in ascending order
    df = df.sort_values("start_alarm")
else:
    print("The start_alarm column in the dataframe is monotonic increasing\n")

Calculating the alarm durations (might be better to do it later so we have less durations to calculate, unnecessary to calculate it for all of the alarms if we don't use them)

Modifying the machine state to have binary values

In [ ]:
window_df = df.groupby("time_window").agg(
    machine_state=("machine_state", "first")#,
    #start_time=("start_alarm", "min")
    ).reset_index()

display(window_df.head())
window_df.info()
#encoding_dict= {"machine_state": {"Failure": 1, "Running": 0}}

#window_df.replace(encoding_dict, inplace=True)
#window_df["machine_state"] = window_df["machine_state"].astype(int)
window_df["machine_state"] = (
    window_df["machine_state"]
    .map({
        "Failure": 1,
        "Running": 0
    })
)

display(window_df.head())
window_df.info()

Adding next state and failure_next (one if the machine fails in the next timeframe) columns

In [ ]:
window_df["next_state"] = (
    window_df["machine_state"]
    .shift(-1)
    .fillna(0)
    .astype(int)
)


display(window_df.head())
window_df.info()
window_df["failure_next"] = (
    (window_df["machine_state"] == 0) &
    (window_df["next_state"] == 1)
).astype(int)

display(window_df.head(10))
window_df.info()

Extracting every instance of the machine failing

In [ ]:
failure_windows = window_df.loc[
    window_df["failure_next"] == 1,
    "time_window"
]

display(failure_windows.head())

Creating a DataFrame which includes only the time_windows of the original df DataFrame, that are followed by failure

In [ ]:
df_failure_windows = df[df["time_window"].isin(failure_windows)]

display(df_failure_windows.head())

Counting how many times each alarm occurred in these time_windows.

In [ ]:
alarm_counts = (
    df_failure_windows["alarm_id"]
    .value_counts()
).reset_index()

display(alarm_counts.head(10))

#most influential alarms
a1 = alarm_counts["alarm_id"].iloc[0]
a2 = alarm_counts["alarm_id"].iloc[1]
a3 = alarm_counts["alarm_id"].iloc[3]


selected_alarms = [a1, a2, a3]
print("The selected alarms are:")
print(selected_alarms)

Extracting the number of times each selected alarm has occurred, and replacing the time_windows in which none of them were present.

In [ ]:
alarm_features = (
    df[df["alarm_id"].isin(selected_alarms)]
      .groupby(["time_window", "alarm_id"])
      .size()
      .unstack(fill_value=0)
)

display(alarm_features.head())
alarm_features.info()

#replacing the windows that didn't include any of the alarms
alarm_features = alarm_features.reindex(df["time_window"].unique(), fill_value=0)

alarm_features = alarm_features.rename(columns={
    a1: "A1_count",
    a2: "A2_count",
    a3: "A3_count"
})

alarm_features = alarm_features.reset_index()

display(alarm_features.tail())
alarm_features.info()

In [ ]:
def discretize_counts(x):
    if x == 0:
        return 0
    elif x <= 2:
        return 1
    elif x <= 5:
        return 2
    else:
        return 3

In [ ]:
alarm_features_discrete = alarm_features.set_index("time_window").apply(
    lambda col: col.map(discretize_counts)
).reset_index()
display(alarm_features_discrete)
alarm_features_discrete.info()

In [ ]:
print(alarm_features_discrete.max(axis=0))

In [ ]:
duration_features = (
    df[df["alarm_id"].isin(selected_alarms)]
      .assign(
          alarm_duration=lambda x:
              (x["end_alarm"] - x["start_alarm"]).dt.total_seconds()
      )
      .groupby(["time_window", "alarm_id"])["alarm_duration"]
      .sum()
      .unstack(fill_value=0)
)

duration_features = duration_features.reindex(
    df["time_window"].unique(),
    fill_value=0
)

duration_features = duration_features.rename(columns={
    a1: "A1_duration",
    a2: "A2_duration",
    a3: "A3_duration"
})

duration_features = duration_features.reset_index()
duration_features

In [ ]:
def discretize_durations(x):
    if x == 0:
        return 0
    elif x <= 30:
        return 1
    elif x <= 300:
        return 2
    else:
        return 3

In [ ]:
duration_features_discrete = duration_features.apply(lambda col: col.map(discretize_durations))
display(duration_features_discrete)
duration_features_discrete.info()

In [ ]:
print(duration_features_discrete.max(axis=0))

In [ ]:
df2 = pd.DataFrame(columns=["time_window", "A1_count", "A1_duration", "A2_count",
                            "A2_duration", "A3_count", "A3_duration", "State_t", "State_t+1" ])

df2["time_window"]=window_df["time_window"]
df2["State_t"]=window_df["machine_state"]
df2["State_t+1"]=window_df["next_state"]
df2["A1_count"]=alarm_features_discrete["A1_count"]
df2["A2_count"]=alarm_features_discrete["A2_count"]
df2["A3_count"]=alarm_features_discrete["A3_count"]
df2["A1_duration"]=duration_features_discrete["A1_duration"]
df2["A2_duration"]=duration_features_discrete["A2_duration"]
df2["A3_duration"]=duration_features_discrete["A3_duration"]
df2

Define the Prediction Target

In [ ]:
target = df2["State_t+1"]
target

Define observation window/feature set

In [ ]:
df_dbn = df2.copy().drop("time_window", axis=1)
df_dbn

In [ ]:
# rename to plain variable names, then assign proper (var, time_slice) tuples

df_dbn = df_dbn.rename(columns={

    "State_t": ("State", 0),

    "State_t+1": ("State", 1),

    "A1_count": ("A1_count", 0),

    "A1_duration": ("A1_duration", 0),

    "A2_count": ("A2_count", 0),

    "A2_duration": ("A2_duration", 0),

    "A3_count": ("A3_count", 0),

    "A3_duration": ("A3_duration", 0),

})

df_dbn.columns = pd.MultiIndex.from_tuples(df_dbn.columns)

In [ ]:
for var in ["A1_count", "A1_duration", "A2_count", "A2_duration", "A3_count", "A3_duration"]:
    df_dbn[(var, 1)] = df_dbn[(var, 0)]

Creating the Dynamic Bayesian Network

In [ ]:
from pgmpy.models import DynamicBayesianNetwork as DBN

dbn = DBN()

In [ ]:
dbn.add_edges_from([
    (("A1_count", 0), ("A1_count", 1)),
    (("A1_duration", 0), ("A1_duration", 1)),
    (("A2_count", 0), ("A2_count", 1)),
    (("A2_duration", 0), ("A2_duration", 1)),
    (("A3_count", 0), ("A3_count", 1)),
    (("A3_duration", 0), ("A3_duration", 1)),

    (("A1_count", 0), ("State", 1)),
    (("A1_duration", 0), ("State", 1)),
    (("A2_count", 0), ("State", 1)),
    (("A2_duration", 0), ("State", 1)),
    (("A3_count", 0), ("State", 1)),
    (("A3_duration", 0), ("State", 1)),
    (("State", 0), ("State", 1)),
])

In [ ]:
list(dbn.nodes())

In [ ]:
dbn.fit(df_dbn) #max likelihood estimator might lead to 0 probabilities

In [ ]:
#print(list(dbn.nodes()))
#print(dbn.edges())
#print([cpd.variable for cpd in dbn.get_cpds()])

In [ ]:
from pgmpy.inference import DBNInference

dbn_infer = DBNInference(dbn)

evidence = {
    ("A1_count", 0): 0, #0-3 None, Low, Medium, High
    ("A1_duration", 0): 0, #0-3 None, Short, Medium, Long
    ("A2_count", 0): 0,
    ("A2_duration", 0): 0,
    ("A3_count", 0): 0,
    ("A3_duration", 0): 1, #0-1
    ("State", 0): 0, #0: Running, 1: Failure
}

In [ ]:
result = dbn_infer.query(variables=[("State", 1)], evidence=evidence)
print(result[("State", 1)])